# Week 2 - Contextual Data Fusion & Feature Engineering
## Contextual Predictive Maintenance (IoT Edge AI)
### Objective: Simulate contextual data, merge with sensor data, engineer enriched features

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Week 1's clean dataset
df = pd.read_csv('../data/clean_dataset.csv')
print(f"Dataset loaded: {df.shape}")
df.head()

Dataset loaded: (10000, 31)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,Torque_rolling_std,Tool_wear_rolling_mean,Tool_wear_rolling_std,Air_temperature_roc,Process_temperature_roc,Rotational_speed_roc,Torque_roc,Tool_wear_roc,power_proxy,temp_delta
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,66382.8,10.5
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,2.474874,1.500000,2.121320,0.1,0.1,-143.0,3.5,3.0,65190.4,10.5
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,3.302020,2.666667,2.516611,-0.1,-0.2,90.0,3.1,2.0,74001.2,10.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,4.287190,3.750000,2.986079,0.1,0.1,-65.0,-9.9,2.0,56603.5,10.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,4.223150,4.800000,3.492850,0.0,0.1,-25.0,0.5,2.0,56320.0,10.5


## 1. Simulate Contextual Data

In [3]:
# Set random seed for reproducibility
np.random.seed(42)

# Simulate Ambient Temperature (factory environment, separate from machine sensors)
df['Ambient_Temperature'] = np.random.normal(loc=25, scale=5, size=len(df))

print("Ambient Temperature simulated!")
df[['Air temperature [K]', 'Ambient_Temperature']].head()

Ambient Temperature simulated!


,Air temperature [K],Ambient_Temperature
0,298.1,27.483571
1,298.2,24.308678
2,298.1,28.238443
3,298.2,32.615149
4,298.2,23.829233


In [4]:
# Simulate Load Density (how heavily loaded the machine is)
# Correlated with Torque - higher torque often means higher load
df['Load_Density'] = (df['Torque [Nm]'] / df['Torque [Nm]'].max()) * 100 + np.random.normal(0, 5, size=len(df))
df['Load_Density'] = df['Load_Density'].clip(0, 100)  # Keep between 0-100%

print("Load Density simulated!")
df[['Torque [Nm]', 'Load_Density']].head()

Load Density simulated!


,Torque [Nm],Load_Density
0,42.8,52.482200
1,46.3,58.916367
2,49.4,61.503956
3,39.5,52.118670
4,40.0,58.205214


In [5]:
# Simulate Environmental Conditions (Humidity %)
df['Humidity'] = np.random.normal(loc=50, scale=10, size=len(df))
df['Humidity'] = df['Humidity'].clip(0, 100)  # Keep between 0-100%

print("Humidity simulated!")
df[['Humidity']].head()

Humidity simulated!


,Humidity
0,53.482862
1,52.833236
2,40.634802
3,55.795842
4,35.099173


In [6]:
# Simulate Operational Context (Shift Timing)
shifts = ['Morning', 'Afternoon', 'Night']
df['Shift'] = np.random.choice(shifts, size=len(df), p=[0.4, 0.35, 0.25])

print("Shift simulated!")
print(df['Shift'].value_counts())

Shift simulated!
Shift
Morning      4044
Afternoon    3410
Night        2546
Name: count, dtype: int64


## 2. Encode Categorical Features

In [7]:
# One-Hot Encode the Shift column
df = pd.get_dummies(df, columns=['Shift'], prefix='Shift')

print("Shift encoded!")
df.filter(like='Shift').head()

Shift encoded!


,Shift_Afternoon,Shift_Morning,Shift_Night
0,False,False,True
1,False,False,True
2,True,False,False
3,False,True,False
4,True,False,False


In [8]:
# One-Hot Encode the Type column (Machine quality: L, M, H)
df = pd.get_dummies(df, columns=['Type'], prefix='Type')

print("Type encoded!")
df.filter(like='Type').head()

Type encoded!


,Type_H,Type_L,Type_M
0,False,False,True
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False


## 3. Contextual Feature Engineering

In [9]:
# Check current dataset shape and columns
print(f"Current shape: {df.shape}")
print(f"\nAll columns:\n{df.columns.tolist()}")

Current shape: (10000, 39)

All columns:
['UDI', 'Product ID', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Air_temperature_rolling_mean', 'Air_temperature_rolling_std', 'Process_temperature_rolling_mean', 'Process_temperature_rolling_std', 'Rotational_speed_rolling_mean', 'Rotational_speed_rolling_std', 'Torque_rolling_mean', 'Torque_rolling_std', 'Tool_wear_rolling_mean', 'Tool_wear_rolling_std', 'Air_temperature_roc', 'Process_temperature_roc', 'Rotational_speed_roc', 'Torque_roc', 'Tool_wear_roc', 'power_proxy', 'temp_delta', 'Ambient_Temperature', 'Load_Density', 'Humidity', 'Shift_Afternoon', 'Shift_Morning', 'Shift_Night', 'Type_H', 'Type_L', 'Type_M']


In [10]:
# Check exact column names
for col in df.columns:
    print(repr(col))

'UDI'
'Product ID'
'Air temperature [K]'
'Process temperature [K]'
'Rotational speed [rpm]'
'Torque [Nm]'
'Tool wear [min]'
'Machine failure'
'TWF'
'HDF'
'PWF'
'OSF'
'RNF'
'Air_temperature_rolling_mean'
'Air_temperature_rolling_std'
'Process_temperature_rolling_mean'
'Process_temperature_rolling_std'
'Rotational_speed_rolling_mean'
'Rotational_speed_rolling_std'
'Torque_rolling_mean'
'Torque_rolling_std'
'Tool_wear_rolling_mean'
'Tool_wear_rolling_std'
'Air_temperature_roc'
'Process_temperature_roc'
'Rotational_speed_roc'
'Torque_roc'
'Tool_wear_roc'
'power_proxy'
'temp_delta'
'Ambient_Temperature'
'Load_Density'
'Humidity'
'Shift_Afternoon'
'Shift_Morning'
'Shift_Night'
'Type_H'
'Type_L'
'Type_M'


In [11]:
print('Ambient_Temperature' in df.columns)
print('Load_Density' in df.columns)
print('Humidity' in df.columns)

True
True
True


In [12]:
# Re-create Ambient Temperature (in case it got lost)
np.random.seed(42)
df['Ambient_Temperature'] = np.random.normal(loc=25, scale=5, size=len(df))
print("Ambient_Temperature recreated!")

Ambient_Temperature recreated!


In [13]:
# Create a fused feature: Temperature Differential
# Difference between machine's internal process temp and ambient room temp
df['Temp_Differential'] = df['Process temperature [K]'] - 273.15 - df['Ambient_Temperature']

print("Temperature Differential created!")
df[['Process temperature [K]', 'Ambient_Temperature', 'Temp_Differential']].head()

Temperature Differential created!


,Process temperature [K],Ambient_Temperature,Temp_Differential
0,308.6,27.483571,7.966429
1,308.7,24.308678,11.241322
2,308.5,28.238443,7.111557
3,308.6,32.615149,2.834851
4,308.7,23.829233,11.720767


In [14]:
# Create a fused feature: Stress Index
# Combines load density and tool wear into a single risk indicator
df['Stress_Index'] = (df['Load_Density'] / 100) * df['Tool wear [min]']

print("Stress Index created!")
df[['Load_Density', 'Tool wear [min]', 'Stress_Index']].head()

Stress Index created!


,Load_Density,Tool wear [min],Stress_Index
0,52.482200,0,0.000000
1,58.916367,3,1.767491
2,61.503956,5,3.075198
3,52.118670,7,3.648307
4,58.205214,9,5.238469


## 4. Save Context-Enriched Dataset

In [15]:
# Save the context-enriched dataset
df.to_csv('../data/context_enriched_dataset.csv', index=False)
print("Context-enriched dataset saved!")
print(f"Final shape: {df.shape}")

Context-enriched dataset saved!
Final shape: (10000, 41)


## 5. Ablation Study - Does Contextual Data Actually Help?

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Modeling libraries imported!")

Modeling libraries imported!


In [17]:
# Define sensor columns (original features only)
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
               'Air_temperature_rolling_mean', 'Air_temperature_rolling_std',
               'Process_temperature_rolling_mean', 'Process_temperature_rolling_std',
               'Rotational_speed_rolling_mean', 'Rotational_speed_rolling_std',
               'Torque_rolling_mean', 'Torque_rolling_std',
               'Tool_wear_rolling_mean', 'Tool_wear_rolling_std',
               'Air_temperature_roc', 'Process_temperature_roc']

# Prepare data for baseline model
X_baseline = df[sensor_cols]
y = df['Machine failure']

# Train-test split
X_train_base, X_test_base, y_train, y_test = train_test_split(X_baseline, y, test_size=0.2, random_state=42, stratify=y)

print(f"Baseline features (sensors only): {len(sensor_cols)}")
print(f"Training set shape: {X_train_base.shape}")
print(f"Test set shape: {X_test_base.shape}")

Baseline features (sensors only): 17
Training set shape: (8000, 17)
Test set shape: (2000, 17)


In [18]:
# Train Baseline Model (sensors only)
print("Training Baseline Model (sensors only)...")
model_baseline = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model_baseline.fit(X_train_base, y_train)

# Evaluate baseline model
y_pred_base = model_baseline.predict(X_test_base)
acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)

print(f"\n=== BASELINE MODEL (Sensors Only) ===")
print(f"Accuracy: {acc_base:.4f}")
print(f"F1 Score: {f1_base:.4f}")

Training Baseline Model (sensors only)...

=== BASELINE MODEL (Sensors Only) ===
Accuracy: 0.9845
F1 Score: 0.7304


In [19]:
# Define contextual features we added
contextual_cols = ['Ambient_Temperature', 'Load_Density', 'Humidity', 
                   'Temp_Differential', 'Stress_Index',
                   'Shift_Morning', 'Shift_Afternoon', 'Shift_Night',
                   'Type_H', 'Type_L', 'Type_M']

# Prepare data for enhanced model (sensors + contextual)
X_enhanced = df[sensor_cols + contextual_cols]

# Use same train-test split
X_train_enh, X_test_enh, _, _ = train_test_split(X_enhanced, y, test_size=0.2, random_state=42, stratify=y)

print(f"Enhanced features (sensors + contextual): {len(sensor_cols + contextual_cols)}")
print(f"Training set shape: {X_train_enh.shape}")
print(f"Test set shape: {X_test_enh.shape}")

Enhanced features (sensors + contextual): 28
Training set shape: (8000, 28)
Test set shape: (2000, 28)


In [20]:
# Train Enhanced Model (sensors + contextual)
print("Training Enhanced Model (sensors + contextual)...")
model_enhanced = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model_enhanced.fit(X_train_enh, y_train)

# Evaluate enhanced model
y_pred_enh = model_enhanced.predict(X_test_enh)
acc_enh = accuracy_score(y_test, y_pred_enh)
f1_enh = f1_score(y_test, y_pred_enh)

print(f"\n=== ENHANCED MODEL (Sensors + Contextual) ===")
print(f"Accuracy: {acc_enh:.4f}")
print(f"F1 Score: {f1_enh:.4f}")

Training Enhanced Model (sensors + contextual)...

=== ENHANCED MODEL (Sensors + Contextual) ===
Accuracy: 0.9780
F1 Score: 0.5686
